In [1]:
import pandas as pd
import numpy as np
import itertools
from joblib import Parallel, delayed
import time

# Data

In [2]:
daily_data = pd.DataFrame()
data_SP500 = pd.read_parquet('/Users/forget/Library/Mobile Documents/com~apple~CloudDocs/Project Stock Market Deep Learning/Data/data_SP500.parquet')
data_NDX = pd.read_parquet('/Users/forget/Library/Mobile Documents/com~apple~CloudDocs/Project Stock Market Deep Learning/Data/data_NASDAQ.parquet')
data_SmallCap = pd.read_parquet('/Users/forget/Library/Mobile Documents/com~apple~CloudDocs/Project Stock Market Deep Learning/Data/data_SmallCap_US.parquet')
data_MidCap = pd.read_parquet('/Users/forget/Library/Mobile Documents/com~apple~CloudDocs/Project Stock Market Deep Learning/Data/data_MidCap_US.parquet')
data_Crypto = pd.read_parquet('/Users/forget/Library/Mobile Documents/com~apple~CloudDocs/Project Stock Market Deep Learning/Data/data_Crypto.parquet')
data_MP = pd.read_parquet('/Users/forget/Library/Mobile Documents/com~apple~CloudDocs/Project Stock Market Deep Learning/Data/data_MP.parquet')
daily_data = pd.concat([data_SP500], ignore_index=True)
daily_data = daily_data.drop_duplicates(subset=['Ticker', 'Date'])
daily_data = daily_data.drop(columns='Adj Close')
daily_data = daily_data[daily_data['Date'] >= '1990-12-31']
daily_data

Price,Date,Close,High,Low,Open,Volume,Ticker
545314,1990-12-31,3.425134,3.770112,3.351211,3.770112,679801.0,NOC
545315,1990-12-31,5.521723,5.521723,5.380141,5.397838,687000.0,MMC
545316,1990-12-31,4.594224,4.594224,4.487382,4.487382,138600.0,AVY
545317,1990-12-31,3.996763,4.065084,3.962603,4.030923,305844.0,APA
545318,1990-12-31,6.815619,6.916218,6.639569,6.790469,525600.0,FDX
...,...,...,...,...,...,...,...
3840744,2025-11-21,219.000000,220.020004,216.470001,218.820007,2475900.0,RSG
3840745,2025-11-21,70.330002,70.529999,68.190002,68.290001,10848400.0,GM
3840746,2025-11-21,79.459999,80.059998,77.389999,78.559998,7036500.0,GLW
3840747,2025-11-21,174.720001,175.380005,169.529999,169.949997,2204000.0,DRI


In [3]:
class OHLCResampler:
    def __init__(self, data, date_col="Date", ticker_col="Ticker"):
        self.data = data.copy()
        self.date_col = date_col
        self.ticker_col = ticker_col

        # S'assurer que la colonne de date est bien en datetime
        self.data[date_col] = pd.to_datetime(self.data[date_col])
        self.data.set_index(date_col, inplace=True)

    def resample(self, period='W'):
        grouped = self.data.groupby(self.ticker_col)

        resampled = grouped.resample(period).agg({
            'Open': 'first',
            'High': 'max',
            'Low': 'min',
            'Close': 'last',
            'Volume': 'sum'
        }).reset_index()

        return resampled

In [4]:
resampler = OHLCResampler(daily_data)

# Period
data = resampler.resample('W')
data

Price,Ticker,Date,Open,High,Low,Close,Volume
0,A,1999-11-21,27.299935,29.999932,23.887440,24.224939,77780526.0
1,A,1999-11-28,24.787442,26.399939,23.999937,24.712440,19126178.0
2,A,1999-12-05,24.599941,27.412429,24.337441,26.699930,17453611.0
3,A,1999-12-12,27.149936,27.862433,26.587434,26.849932,10048404.0
4,A,1999-12-19,27.299925,28.349922,24.599949,27.562433,15559180.0
...,...,...,...,...,...,...,...
683238,ZTS,2025-10-26,143.869995,148.300003,143.509995,145.940002,10369600.0
683239,ZTS,2025-11-02,146.021567,147.416703,141.945762,144.089996,14252100.0
683240,ZTS,2025-11-09,143.369995,144.830002,117.260002,120.239998,37341000.0
683241,ZTS,2025-11-16,120.430000,122.959999,118.519997,120.820000,22197200.0


# Backtest

### Generate Signals

In [5]:
def generate_signals(df, momentum_window, min_momentum, top_n, ma_fast_window, ma_slow_window, rebalance_freq='W'):
    """
    rebalance_freq : 'W' (Weekly - ton code actuel), 'M' (Mensuel), 'Q' (Trimestriel)
    """
    data = df.copy()
    data = data.sort_values(['Ticker', 'Date'])

    # 1. Indicateurs Techniques (Calculés partout pour avoir l'historique)
    data['Momentum'] = data.groupby('Ticker')['Close'].pct_change(periods=momentum_window)

    # Moyennes Mobiles
    data['ma_fast'] = data.groupby('Ticker')['Close'].transform(lambda x: x.rolling(ma_fast_window).mean())
    data['ma_slow'] = data.groupby('Ticker')['Close'].transform(lambda x: x.rolling(ma_slow_window).mean())

    # Filtre de tendance
    data['Trend_Filter'] = data['ma_slow'] < data['ma_fast']

    # Rendement futur
    data['NextReturn'] = data.groupby('Ticker')['Close'].pct_change().shift(-1)

    # --- 2. GESTION DU REBALANCEMENT ---

    # Si fréquence hebdomadaire ('W'), on garde le comportement glissant classique
    if rebalance_freq == 'W':
        is_rebalance_date = pd.Series(True, index=data.index)
    else:
        # Astuce Pandas pour trouver la dernière date dispo de chaque période
        # On crée une colonne 'Period' (ex: '2020-01' pour Janvier)
        data['Period'] = data['Date'].dt.to_period(rebalance_freq)

        # On identifie la dernière date présente dans tes données pour chaque période
        rebalance_dates = data.groupby('Period')['Date'].transform('max')

        # Masque booléen : True seulement si c'est la date de rebalancement
        is_rebalance_date = (data['Date'] == rebalance_dates)

    # --- 3. RANKING (Uniquement sur les dates de rebalancement) ---

    # On met -inf partout où ce n'est pas une date de rebalance OU conditions non remplies
    # Cela force le ranking à ignorer les semaines intermédiaires pour l'instant
    valid_momentum = data['Momentum'].where(
        (data['Momentum'] > min_momentum) & (data['Trend_Filter']) & (is_rebalance_date),
        other=np.NINF
    )

    # Ranking
    data['Rank'] = valid_momentum.groupby(data['Date']).rank(method='first', ascending=False)

    # Signal brut (1 seulement les jours de rebalancement, 0 sinon)
    # Note: On met NaN au lieu de 0 pour les jours sans rebalance afin de pouvoir faire un ffill()
    data['Raw_Signal'] = np.where(
        is_rebalance_date,
        np.where((data['Rank'] <= top_n) & (valid_momentum > np.NINF), 1, 0),
        np.nan
    )

    # PROPAGATION DU SIGNAL (Forward Fill)
    # On répète la décision prise lors du rebalancement sur les semaines suivantes
    data['Prediction'] = data.groupby('Ticker')['Raw_Signal'].ffill().fillna(0)

    # Nettoyage
    data = data.dropna(subset=['NextReturn', 'Momentum'])

    return data

### Vectorized Backtester

In [6]:
class VectorizedBacktester:
    def __init__(self, processed_data, initial_capital=10000, fee_per_trade=1, top_n=5):
        self.df = processed_data
        self.initial_capital = initial_capital
        self.fees = fee_per_trade
        self.top_n = top_n
        self.dates = self.df['Date'].drop_duplicates().sort_values().to_numpy()

    def run(self):
        capital = self.initial_capital
        holdings = set()
        history_capital = []

        # Conversion du DF en dict de dicts ou numpy arrays pour vitesse maximale dans la boucle
        # Ici, on reste en pandas optimisé par date pour la lisibilité,
        # mais on extrait les arrays numpy pour les calculs

        for date in self.dates:
            # Slice du jour (rapide si indexé, sinon boolean mask)
            day_data = self.df[self.df['Date'] == date]

            if day_data.empty:
                history_capital.append(capital)
                continue

            # Identification des signaux
            # Actions qui devraient être dans le portefeuille
            target_holdings = set(day_data[day_data['Prediction'] == 1]['Ticker'])

            # Logique de rebalancement
            to_buy = target_holdings - holdings
            to_sell = holdings - target_holdings

            # Si une action n'est plus dans le top_n ou ne satisfait plus les conditions, on vend.
            # Note: Ici on vend tout ce qui n'est pas "Target".
            # On pourrait ajouter une logique de "Buffer" pour réduire le turnover.

            n_trades = len(to_buy) + len(to_sell)
            capital -= (n_trades * self.fees)

            # Mise à jour des positions détenues
            holdings = (holdings - to_sell).union(to_buy)

            # Calcul du rendement de la période
            if holdings:
                # Poids équi-réparti dynamique
                weight = 1.0 / self.top_n

                # Récupération des rendements futurs des actions détenues
                relevant_returns = day_data[day_data['Ticker'].isin(holdings)]['NextReturn'].values

                # Rendement du portefeuille = Somme pondérée
                port_return = np.sum(relevant_returns) * weight
                capital *= (1 + port_return)

            history_capital.append(capital)

        # Création des résultats
        results = pd.DataFrame({'Date': self.dates, 'Capital': history_capital}).set_index('Date')
        return results

    def get_metrics(self, results):
        """Calcule les métriques clés pour la Grid Search"""
        initial = results['Capital'].iloc[0]
        final = results['Capital'].iloc[-1]

        # Returns
        results['Returns'] = results['Capital'].pct_change().fillna(0)
        total_return = (final - initial) / initial

        # CAGR
        n_years = (results.index[-1] - results.index[0]).days / 365.25
        cagr = (final / initial) ** (1 / n_years) - 1 if n_years > 0 else 0

        # Drawdown
        rolling_max = results['Capital'].cummax()
        drawdown = (results['Capital'] - rolling_max) / rolling_max
        max_dd = drawdown.min()

        # Sharpe (hebdomadaire annualisé)
        # Supposons 52 semaines
        mean_ret = results['Returns'].mean()
        std_ret = results['Returns'].std()
        sharpe = (mean_ret / std_ret) * np.sqrt(52) if std_ret > 0 else 0

        return {
            "Total Return": total_return,
            "CAGR": cagr,
            "Max Drawdown": max_dd,
            "Sharpe Ratio": sharpe,
            "Final Capital": final
        }

In [7]:
def run_single_backtest(params, df_source, start_date, end_date):
    # Ajout de rebalance_freq dans le déballage des params
    mom_win, min_mom, top_n, ma_fast, ma_slow, reb_freq = params

    default_output = {
        "Momentum_Window": mom_win,
        "Min_Momentum": min_mom,
        "Top_N": top_n,
        "MA_Fast": ma_fast,
        "MA_Slow": ma_slow,
        "Rebalance": reb_freq, # Ajouté aux logs
        "Sharpe Ratio": np.nan,
        "Total Return": np.nan,
        "Max Drawdown": np.nan,
        "Error": None
    }

    try:
        # Appel avec le nouveau paramètre
        full_signals = generate_signals(
            df_source,
            momentum_window=mom_win,
            min_momentum=min_mom,
            top_n=top_n,
            ma_fast_window=ma_fast,
            ma_slow_window=ma_slow,
            rebalance_freq=reb_freq # <--- ICI
        )

        # Filtre de date (identique à avant)
        mask = (full_signals['Date'] >= pd.to_datetime(start_date)) & \
               (full_signals['Date'] <= pd.to_datetime(end_date))
        backtest_data = full_signals.loc[mask].copy()

        if backtest_data.empty:
            default_output["Error"] = "No data"
            return default_output

        # Backtest (Le Backtester ne change pas, il reçoit juste des signaux qui changent moins souvent)
        bt = VectorizedBacktester(backtest_data, initial_capital=10000, fee_per_trade=1, top_n=top_n)
        res_df = bt.run()
        metrics = bt.get_metrics(res_df)

        output = default_output.copy()
        output.update(metrics)
        return output

    except Exception as e:
        default_output["Error"] = str(e)
        return default_output

In [8]:
def grid_search_execution(df, param_grid, start_date, end_date):
    # Création des combinaisons
    keys, values = zip(*param_grid.items())
    combinations = [v for v in itertools.product(*values)]

    print(f"Lancement de la Grid Search sur {len(combinations)} combinaisons...")
    print(f"Période de test : {start_date} à {end_date}")

    start_time = time.time()

    # On passe start_date et end_date dans arguments de delayed()
    results_list = Parallel(n_jobs=-1)(
        delayed(run_single_backtest)(params, df, start_date, end_date) for params in combinations
    )

    end_time = time.time()
    print(f"Terminé en {end_time - start_time:.2f} secondes.")

    results_df = pd.DataFrame(results_list)
    return results_df

In [9]:
# Simulation de données pour l'exemple (À remplacer par ton 'data')
# Assure-toi que 'data' contient : Date, Ticker, Close
# data['Date'] = pd.to_datetime(data['Date'])

# Définition de la grille de paramètres
param_grid = {
    'momentum_window': [12, 24, 52],      # 1 mois, 3 mois, 6 mois (en semaines)
    'min_momentum': np.arange(0.2, 0.6, 0.05),  # 10%, 20%, 30%
    'top_n': np.arange(1, 15),                # Diversification
    'ma_fast': [12, 25],                     # On peut fixer certaines variables
    'ma_slow': [50, 100],
    'rebalance_freq': ['W', 'Q', 'M', '6M']
}

# Lancement
results_df = grid_search_execution(
    df=data,
    param_grid=param_grid,
    start_date='2000-01-01',
    end_date='2026-01-01'
)

Lancement de la Grid Search sur 5376 combinaisons...
Période de test : 2000-01-01 à 2026-01-01


/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_1644/587077851.py:9: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_1644/587077851.py:9: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_1644/587077851.py:9: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
/var

Terminé en 1525.27 secondes.


In [10]:
results_df.sort_values(by="CAGR", ascending=False).head(50)

,Momentum_Window,Min_Momentum,Top_N,MA_Fast,MA_Slow,Rebalance,Sharpe Ratio,Total Return,Max Drawdown,Error,CAGR,Final Capital
1813,24,0.20,2,12,100,Q,1.066025,33872.223848,-0.543581,None,0.496520,2.948264e+08
4732,52,0.45,2,25,100,W,1.092218,32906.822250,-0.614545,None,0.494849,2.864237e+08
4284,52,0.35,2,25,100,W,1.090475,32706.771878,-0.614545,None,0.494496,2.846825e+08
5180,52,0.55,2,25,100,W,1.092097,32318.357452,-0.614545,None,0.493806,2.813019e+08
2037,24,0.25,2,12,100,Q,1.060493,31311.202218,-0.543581,None,0.491980,2.725358e+08
4508,52,0.40,2,25,100,W,1.085751,30716.447023,-0.614545,None,0.490874,2.673591e+08
4956,52,0.50,2,25,100,W,1.087087,30617.917744,-0.614545,None,0.490689,2.665015e+08
5172,52,0.55,2,12,100,W,1.085676,29774.998340,-0.614545,None,0.489082,2.591649e+08
3610,52,0.20,2,25,50,M,1.065765,28774.937151,-0.708564,None,0.487117,2.504606e+08
3611,52,0.20,2,25,50,6M,1.065765,28774.937151,-0.708564,None,0.487117,2.504606e+08


In [12]:
results_df.to_csv("Results_Momentum_Fixed_Start2000_to_2026.csv")